# Backtesting Intro – zipline-reloaded\n\nThis notebook demonstrates a simple SMA crossover strategy on AAPL data using the zipline-reloaded bundle we ingested earlier.

In [ ]:
%matplotlib inline\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom zipline.api import order_target, symbol, set_slippage, set_commission, slippage, commission\nfrom zipline import run_algorithm\nfrom zipline.pipeline import USEquityPricing\nfrom zipline.pipeline.factors import SimpleMovingAverage\nfrom zipline.filters import static\n

In [ ]:
def initialize(context):\n    # 50-day and 200-day SMAs\n    context.sma_50 = SimpleMovingAverage(inputs=[USEquityPricing.close], window_length=50, mask=static())\n    context.sma_200 = SimpleMovingAverage(inputs=[USEquityPricing.close], window_length=200, mask=static())\n    context.asset = symbol('AAPL')\n    set_slippage(slippage.FixedSlippage(spread=0))\n    set_commission(commission.PerShare(cost=0.005))\n

In [ ]:
def handle_data(context, data):\n    if not data.can_trade(context.asset):\n        return\n    sma_50 = context.sma_50[data.date][context.asset]\n    sma_200 = context.sma_200[data.date][context.asset]\n    if np.isnan(sma_50) or np.isnan(sma_200):\n        return\n    if sma_50 > sma_200:\n        target = 1.0  # fully invested\n    else:\n        target = 0.0  # out of market\n    order_target(context.asset, target * context.portfolio.portfolio_value / data.current(context.asset, 'price'))\n

In [ ]:
# Run the algorithm\nstart = pd.Timestamp('2015-01-01', tz='utc')\nend   = pd.Timestamp('2024-12-31', tz='utc')\n\nresult = run_algorithm(\n    start=start,\n    end=end,\n    initialize=initialize,\n    handle_data=handle_data,\n    capital_base=10_000,\n    bundle='raats_daily'\n)\n\n# Extract equity curve\nequity = result.portfolio_value\n\n# Plot\nplt.figure(figsize=(12,6))\nplt.plot(equity.index, equity.values)\nplt.title('Equity Curve – SMA 50/200 Crossover on AAPL (zipline)')\nplt.ylabel('Portfolio Value ($)')\nplt.xlabel('Date')\nplt.grid(True)\nplt.show()\n\n# Print final value and return\nfinal_value = equity.iloc[-1]\ntotal_return = (final_value / 10_000) - 1\nprint(f'Final portfolio value: ${final_value:,.2f}')\nprint(f'Total return: {total_return:.2%}')\n